# Build a Vanilla LLM, End to End

This single notebook builds a small decoder-only Transformer language model from
scratch in PyTorch, then actually trains it on a tiny corpus and samples text
from it. Every core building block is implemented by hand: no `nn.Linear`, no
`nn.MultiheadAttention`, no `F.scaled_dot_product_attention` in the final model.

The notebook is meant to be read and run top to bottom in one kernel. Each block
you build stays in memory and is reused by every block that follows, so by the
end you have assembled the whole model out of your own pieces.

## How each section is structured

For every component you get:

- **What & why** — what the block does and where it sits in the model.
- **Math / algorithm** — the equations, written so you can translate them to code.
- **Design choices & alternatives** — what else people use and the trade-offs.
- **Pitfalls** — the bugs that actually bite when you implement this.
- **Interview angles** — what an interviewer tends to probe on this topic.
- **Your turn** — a stub to implement.
- **Solution — run to continue** — a correct reference so the pipeline keeps working.

## The road map

1. **Primitives** — linear, embedding, SiLU, softmax, RMSNorm, SwiGLU, cross-entropy.
2. **The Transformer** — attention, RoPE, multi-head attention, a pre-norm block, the full LM.
3. **Tokenization** — train a byte-level BPE tokenizer and encode/decode text.
4. **Training systems** — cosine LR schedule, gradient clipping, AdamW, batching, checkpoints.
5. **Train a tiny model** — wire it all together and run a real (tiny) training loop.
6. **Inference** — KV cache, sampling (temperature / top-k / top-p), the generation loop, then sample from the model you trained.
7. **Alignment (bonus)** — SFT loss with prompt masking and the DPO preference loss.

> The reference implementations here mirror the per-component challenges in this
> repo (`notebooks/01`–`06` and `solutions/`). Tests still load those; this
> notebook is the guided, runnable tour.

## Setup

One import cell for the whole notebook. Everything downstream reuses these.

In [ ]:
import io
import math
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

try:
    import regex as re  # \p{L}-style Unicode classes need the `regex` package
except ImportError:  # pragma: no cover
    import re

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__)

# 1. Primitives

The numerically careful building blocks every later section leans on. None of
these are hard individually; the point is to get the shapes, the numerics, and
the conventions right, because every bug here shows up amplified later.

### Linear (no bias)


**What & why.** A linear layer computes $y = x W^\top$. By convention (following
CS336 / most HF code) the weight is stored as `(d_out, d_in)`, so the forward
pass transposes once. Modern LLMs (Llama, Mistral, Qwen, GPT-NeoX) drop the bias
on essentially every linear, including Q/K/V projections and the LM head. This is
the most reused primitive in the whole model.

**Math.**
$$y = x W^\top, \qquad W \in \mathbb{R}^{d_\text{out} \times d_\text{in}}$$

**Design choices & alternatives.**
- *Bias vs no bias.* Biases add parameters and memory traffic for little quality
  gain at scale, and they slightly complicate tensor-parallel sharding. Most LLMs
  drop them.
- *Storage layout.* `(d_out, d_in)` lets the LM head share the embedding matrix
  (weight tying) without a transpose mismatch.
- *Fused kernels.* In production the matmul is fused (cuBLAS / Triton); here we
  just call `@`.

**Pitfalls.**
- Forgetting that leading batch dims must be preserved — `x` can be `(..., d_in)`.
- Transposing the wrong axes for batched weights; use `transpose(-1, -2)`, not `.T`.

**Interview angles.**
- Why store weights as `(d_out, d_in)`? (transpose-free LM-head tying, matmul layout)
- What is the FLOP count of a linear layer? ($2 \cdot d_\text{in} \cdot d_\text{out}$ per token)
- Why do modern LLMs drop bias terms?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def linear(d_in, d_out, weights, in_features):
    # weights: (d_out, d_in); in_features: (..., d_in) -> (..., d_out)
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def linear(d_in, d_out, weights, in_features):
    return in_features @ weights.transpose(-1, -2)


_x = torch.randn(2, 3, 5)
_w = torch.randn(7, 5)
assert torch.allclose(linear(5, 7, _w, _x), F.linear(_x, _w), atol=1e-5)
print("linear OK", linear(5, 7, _w, _x).shape)

### Token Embedding


**What & why.** Maps integer token ids to dense vectors via a lookup table of
shape `(vocab_size, d_model)`. Conceptually it is a one-hot matmul, but you always
implement it as an index/gather because that is far cheaper.

**Math.** $\text{emb}(t) = \text{one\_hot}(t)\, W \equiv W[t]$.

**Design choices & alternatives.**
- *Weight tying.* The embedding matrix and the LM head can share weights, saving
  `vocab_size * d_model` parameters. We keep them untied here for clarity.
- *Factorized embeddings* (ALBERT) decompose the table into two smaller matrices
  when the vocab is huge.

**Pitfalls.**
- Indexing with a non-`long` tensor.
- Losing the leading dims of `token_ids` — `W[ids]` preserves them automatically.

**Interview angles.**
- Why is embedding a gather and not a real matmul? (sparsity of one-hot)
- What is weight tying and why does it help? (parameter count, regularization)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def embedding(vocab_size, d_model, weights, token_ids):
    # weights: (vocab_size, d_model); token_ids: (...) int -> (..., d_model)
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def embedding(vocab_size, d_model, weights, token_ids):
    return weights[token_ids]


_w = torch.randn(100, 16)
_ids = torch.tensor([[1, 5, 99], [0, 42, 7]])
assert torch.allclose(embedding(100, 16, _w, _ids)[0, 0], _w[1])
print("embedding OK", embedding(100, 16, _w, _ids).shape)

### SiLU Activation


**What & why.** $\mathrm{silu}(x) = x\,\sigma(x)$. Smooth and non-monotonic; the
gate activation inside SwiGLU. Empirically beats ReLU in Transformer FFNs.

**Math.** $\mathrm{silu}(x) = \dfrac{x}{1 + e^{-x}}$.

**Design choices & alternatives.**
- *ReLU* — cheaper, but a hard zero kills gradient for negative inputs.
- *GELU* — very close to SiLU; used by GPT-2/BERT. SiLU/SwiGLU is the modern default.
- *Squared ReLU* — used by some recent models; cheaper than SiLU.

**Pitfalls.**
- Recomputing `sigmoid` twice instead of reusing it (minor, but a tell).

**Interview angles.**
- Why non-monotonic activations? (small negative pass-through preserves gradient)
- SiLU vs GELU vs ReLU trade-offs.

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def silu(in_features):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def silu(in_features):
    return in_features * torch.sigmoid(in_features)


_x = torch.linspace(-3, 3, 7)
assert torch.allclose(silu(_x), F.silu(_x))
print("silu OK")

### Numerically Stable Softmax


**What & why.** Turns logits into a probability distribution. The naive
`exp(x)/exp(x).sum()` overflows for large logits, so we subtract the max along the
softmax axis first. Softmax is shift-invariant, so this changes nothing
mathematically but keeps every `exp` input $\le 0$.

**Math.**
$$m = \max_i x_i, \qquad y_i = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}$$

**Design choices & alternatives.**
- *log-softmax* is preferred whenever you immediately take a log (e.g. cross-entropy),
  because it avoids a redundant `exp`/`log` round trip.
- *Temperature* scaling ($x/T$) is applied before softmax at sampling time.

**Pitfalls.**
- Forgetting `keepdim=True` on the max so it broadcasts back.
- Hard-coding `dim=-1`; softmax can run over any axis (attention uses the key axis).

**Interview angles.**
- Why subtract the max? Prove it does not change the output. (shift invariance)
- When would you use log-softmax instead? (numerical stability with downstream log)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def softmax(in_features, dim):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def softmax(in_features, dim):
    m, _ = in_features.max(dim=dim, keepdim=True)
    z = torch.exp(in_features - m)
    return z / z.sum(dim=dim, keepdim=True)


_x = torch.tensor([1000., 1001., 1002.])  # would overflow naively
assert torch.isfinite(softmax(_x, dim=-1)).all()
assert torch.allclose(softmax(_x, dim=-1), torch.softmax(_x, dim=-1))
print("softmax OK")

### RMSNorm


**What & why.** RMSNorm replaces LayerNorm in modern LLMs. It rescales each vector
by its root-mean-square and applies a learned per-channel gain — no mean
subtraction, no bias. Cheaper than LayerNorm with no measurable quality loss.

**Math.**
$$\mathrm{rms}(x) = \sqrt{\tfrac{1}{d}\sum_i x_i^2 + \epsilon}, \qquad
y = \frac{x}{\mathrm{rms}(x)} \odot g$$

**Design choices & alternatives.**
- *LayerNorm* centers (subtract mean) and scales; RMSNorm drops the centering.
- *Pre-norm vs post-norm.* Modern LLMs normalize the input to each sub-layer
  (pre-norm) for stable deep training. We use pre-norm in the block.
- *fp32 reduction.* Compute the RMS in fp32 even on fp16/bf16 inputs, then cast
  back, or low-precision training drifts.

**Pitfalls.**
- Reducing over the wrong axis — normalize over the feature dim (`-1`).
- Doing the square/mean in fp16 and losing precision.

**Interview angles.**
- RMSNorm vs LayerNorm — what is dropped and why is it fine?
- Why pre-norm for deep Transformers? (gradient flow, no warmup-free divergence)
- Why upcast to fp32 for the reduction?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def rmsnorm(d_model, eps, weights, in_features):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def rmsnorm(d_model, eps, weights, in_features):
    in_dtype = in_features.dtype
    x = in_features.to(torch.float32)
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    out = (x / rms) * weights.to(torch.float32)
    return out.to(in_dtype)


_d, _eps = 32, 1e-5
_w, _x = torch.randn(_d), torch.randn(2, 4, _d)
assert torch.allclose(rmsnorm(_d, _eps, _w, _x),
                      F.rms_norm(_x, (_d,), weight=_w, eps=_eps), atol=1e-5)
print("rmsnorm OK")

### SwiGLU Feed-Forward


**What & why.** The post-Llama default FFN. Three linears, no bias, no dropout. A
SiLU-gated linear unit: one projection is gated by the SiLU of another, then
projected back down.

**Math.**
$$\mathrm{ffn}(x) = W_2\big(\mathrm{silu}(W_1 x) \odot (W_3 x)\big)$$

**Design choices & alternatives.**
- *Vanilla FFN* (`W_2 (\mathrm{relu}(W_1 x))`) is the original Transformer FFN —
  two matrices, half the gate cost.
- *GeGLU* swaps SiLU for GELU. Gated variants generally win.
- *Width.* Because SwiGLU has three matrices, `d_ff` is usually set to about
  $\tfrac{8}{3} d_\text{model}$ so the parameter count matches a 4× vanilla FFN.

**Pitfalls.**
- Mixing up gate ($W_1$) and up ($W_3$) projections — they are symmetric in shape
  but not interchangeable once trained.
- Reimplementing SiLU instead of reusing the one you just built.

**Interview angles.**
- Why three matrices, and how do you pick `d_ff`?
- What does the gate buy you over a plain FFN? (input-dependent feature selection)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def swiglu(d_model, d_ff, w1_weight, w2_weight, w3_weight, in_features):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def swiglu(d_model, d_ff, w1_weight, w2_weight, w3_weight, in_features):
    gate = silu(in_features @ w1_weight.transpose(-1, -2))
    up = in_features @ w3_weight.transpose(-1, -2)
    return (gate * up) @ w2_weight.transpose(-1, -2)


_dm, _dff = 16, 32
_w1, _w2, _w3 = torch.randn(_dff, _dm), torch.randn(_dm, _dff), torch.randn(_dff, _dm)
_x = torch.randn(2, 7, _dm)
_ref = (F.silu(_x @ _w1.T) * (_x @ _w3.T)) @ _w2.T
assert torch.allclose(swiglu(_dm, _dff, _w1, _w2, _w3, _x), _ref, atol=1e-5)
print("swiglu OK")

### Numerically Stable Cross-Entropy


**What & why.** The pretraining loss: negative log-likelihood of the correct next
token. Compute it directly from logits with the log-sum-exp identity instead of
softmax-then-log, which loses precision.

**Math.**
$$\ell_b = \mathrm{logsumexp}(z_b) - z_{b, t_b}, \qquad
\mathcal{L} = \frac{1}{B}\sum_b \ell_b$$

**Design choices & alternatives.**
- *Label smoothing* spreads a little mass to non-target classes; common in older
  setups, less so for LLM pretraining.
- *Z-loss* adds a small penalty on `logsumexp` to keep logits from drifting.
- *Reduction.* `mean` over tokens is standard; `sum` plus manual normalization is
  used for masked losses (see the SFT section).

**Pitfalls.**
- Passing already-softmaxed probabilities instead of raw logits.
- Subtracting the max but forgetting to add it back into `logsumexp`.

**Interview angles.**
- Derive the gradient of cross-entropy w.r.t. logits. ($\mathrm{softmax}(z) - \text{one\_hot}(t)$)
- Why fuse log-softmax with NLL instead of computing them separately?
- What does perplexity equal in terms of this loss? ($e^{\mathcal{L}}$)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def cross_entropy(inputs, targets):
    # inputs: (batch, vocab) logits; targets: (batch,) class ids -> scalar
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def cross_entropy(inputs, targets):
    m, _ = inputs.max(dim=-1, keepdim=True)
    shifted = inputs - m
    log_z = m.squeeze(-1) + torch.log(torch.exp(shifted).sum(dim=-1))
    target_logits = inputs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return (log_z - target_logits).mean()


_logits = torch.randn(8, 100)
_targets = torch.randint(0, 100, (8,))
assert torch.allclose(cross_entropy(_logits, _targets),
                      F.cross_entropy(_logits, _targets), atol=1e-5)
print("cross_entropy OK")

# 2. The Transformer

Now assemble attention, positional encoding, and the decoder block into a full
language model forward pass. Everything here reuses the primitives above.

### Scaled Dot-Product Attention


**What & why.** The core mixing operation: each query attends to keys, and the
softmax-weighted sum of values is the output. The $1/\sqrt{d_k}$ scale keeps the
dot products from saturating the softmax as the head dimension grows.

**Math.**
$$\mathrm{attn}(Q,K,V) = \mathrm{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}} + M\Big) V$$
where the mask $M$ sets disallowed positions to $-\infty$.

**Design choices & alternatives.**
- *FlashAttention* computes the same result without materializing the
  `(queries, keys)` score matrix, which is the memory bottleneck.
- *MQA / GQA* share K/V across heads to shrink the KV cache at inference.
- *Masking value.* Use `float('-inf')`, not `-1e9`: the latter leaks probability
  in fp16.

**Pitfalls.**
- Scaling by `d_k` instead of `sqrt(d_k)`.
- Masking convention: here `True` means "allowed". Inverting it silently breaks causality.
- Forgetting that the softmax is over the **key** axis, not the query axis.

**Interview angles.**
- Why divide by $\sqrt{d_k}$? (variance of the dot product grows linearly in $d_k$)
- Time and memory complexity in sequence length? ($O(n^2)$ both, $O(n)$ with Flash for memory)
- What is the causal mask and why is it lower-triangular?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q,K: (..., seq, d_k); V: (..., seq, d_v); mask: (..., q, k) bool, True=attend
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = softmax(scores, dim=-1)
    return torch.matmul(weights, V)


_Q, _K, _V = (torch.randn(1, 5, 8) for _ in range(3))
_m = torch.tril(torch.ones(5, 5, dtype=torch.bool))
_ref = F.scaled_dot_product_attention(_Q, _K, _V, is_causal=True)
assert torch.allclose(scaled_dot_product_attention(_Q, _K, _V, mask=_m), _ref, atol=1e-4)
print("attention OK")

### Rotary Position Embeddings (RoPE)


**What & why.** RoPE injects position by *rotating* pairs of dimensions of Q and K
by an angle proportional to position. The attention dot product
$Q_m K_n^\top$ then depends only on the relative offset $m - n$, which is exactly
what you want for length generalization.

**Math.** Frequencies $\theta_i = \text{base}^{-2i/d_k}$ for $i = 0,\dots,d_k/2-1$;
angle $\phi_{p,i} = p\,\theta_i$; rotate each 2-D pair:
$$\begin{pmatrix} x'_{2i} \\ x'_{2i+1}\end{pmatrix} =
\begin{pmatrix}\cos\phi & -\sin\phi \\ \sin\phi & \cos\phi\end{pmatrix}
\begin{pmatrix} x_{2i} \\ x_{2i+1}\end{pmatrix}$$

**Design choices & alternatives.**
- *Sinusoidal* (original Transformer) — added to embeddings, absolute.
- *Learned absolute* (GPT-2) — a position table; does not extrapolate past training length.
- *ALiBi* — a linear bias on attention scores; also relative, simpler.
- *RoPE scaling* (NTK / linear / YaRN) extends context length at inference.

**Pitfalls.**
- Applying RoPE to V (only Q and K get rotated).
- Pairing convention: rotate `(2i, 2i+1)` pairs consistently for Q and K.
- A correct RoPE preserves each token vector's norm — use that as a sanity check.

**Interview angles.**
- Why does RoPE give *relative* position from a per-token rotation?
- RoPE vs ALiBi vs learned embeddings — which extrapolate and why?
- How do you extend a RoPE model to longer context? (frequency scaling)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def rope(d_k, theta, max_seq_len, in_query_or_key, token_positions):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def rope(d_k, theta, max_seq_len, in_query_or_key, token_positions):
    assert d_k % 2 == 0
    device, dtype = in_query_or_key.device, in_query_or_key.dtype
    half = d_k // 2
    inv_freq = theta ** (-torch.arange(0, half, dtype=torch.float32, device=device) * 2.0 / d_k)
    positions = torch.arange(max_seq_len, dtype=torch.float32, device=device)
    angles = torch.einsum("p,h->ph", positions, inv_freq)
    cos = torch.cos(angles)[token_positions]
    sin = torch.sin(angles)[token_positions]
    x = in_query_or_key
    pairs = x.reshape(*x.shape[:-1], half, 2)
    x_even, x_odd = pairs[..., 0], pairs[..., 1]
    out = torch.stack((x_even * cos - x_odd * sin, x_even * sin + x_odd * cos), dim=-1)
    return out.reshape_as(x).to(dtype)


_x = torch.randn(8, 16)
assert torch.allclose(rope(16, 10000.0, 64, _x, torch.arange(8)).norm(dim=-1),
                      _x.norm(dim=-1), atol=1e-4)
print("rope OK")

### Multi-Head Self-Attention (with RoPE + causal mask)


**What & why.** Project the input into Q, K, V, split into `num_heads` independent
heads, run scaled dot-product attention per head with a causal mask, concatenate,
and project back. Multiple heads let the model attend to different relationships
in parallel subspaces. We build the RoPE + causal variant because that is what the
LM block uses.

**Math.** Per head $h$: $\mathrm{attn}(Q_h, K_h, V_h)$ over `head_dim = d_model / num_heads`,
then concat over heads and apply the output projection $W_o$.

**Design choices & alternatives.**
- *MHA vs MQA vs GQA.* MQA shares one K/V head across all query heads (tiny KV cache);
  GQA groups query heads to share K/V — the Llama-2/3 default.
- *Fused QKV.* One `(3·d_model, d_model)` projection is common; we keep three for clarity.

**Pitfalls.**
- Reshape/transpose order when splitting heads: `(..., seq, H, head_dim)` then
  swap to `(..., H, seq, head_dim)`.
- Building the causal mask on the wrong device.
- Applying RoPE after splitting heads (correct) vs before (wrong dim).

**Interview angles.**
- Why multiple heads instead of one big attention? (multiple representation subspaces)
- MQA/GQA: what do they trade and why does it matter at inference? (KV-cache bandwidth)
- Where does RoPE get applied in the multi-head pipeline?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def multihead_self_attention_with_rope(d_model, num_heads, max_seq_len, theta,
                                       q_w, k_w, v_w, o_w, in_features,
                                       token_positions=None):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def _split_heads(x, num_heads):
    *prefix, seq, d_model = x.shape
    return x.reshape(*prefix, seq, num_heads, d_model // num_heads).transpose(-3, -2)


def _merge_heads(x):
    *prefix, num_heads, seq, head_dim = x.shape
    return x.transpose(-3, -2).reshape(*prefix, seq, num_heads * head_dim)


def _causal_mask(seq_len, device):
    return torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))


def multihead_self_attention_with_rope(d_model, num_heads, max_seq_len, theta,
                                       q_w, k_w, v_w, o_w, in_features,
                                       token_positions=None):
    Q = _split_heads(in_features @ q_w.transpose(-1, -2), num_heads)
    K = _split_heads(in_features @ k_w.transpose(-1, -2), num_heads)
    V = _split_heads(in_features @ v_w.transpose(-1, -2), num_heads)
    seq_len = in_features.shape[-2]
    head_dim = d_model // num_heads
    if token_positions is None:
        token_positions = torch.arange(seq_len, device=in_features.device).expand(
            *in_features.shape[:-2], seq_len)
    pos = token_positions.unsqueeze(-2).expand(*Q.shape[:-1])
    Q = rope(head_dim, theta, max_seq_len, Q, pos)
    K = rope(head_dim, theta, max_seq_len, K, pos)
    out = scaled_dot_product_attention(Q, K, V, mask=_causal_mask(seq_len, in_features.device))
    return _merge_heads(out) @ o_w.transpose(-1, -2)


_dm, _h, _seq = 16, 4, 7
_x = torch.randn(2, _seq, _dm)
_ws = [torch.randn(_dm, _dm) / 4 for _ in range(4)]
_o = multihead_self_attention_with_rope(_dm, _h, 64, 10000.0, *_ws, _x)
assert _o.shape == (2, _seq, _dm)
print("multihead OK", _o.shape)

### Pre-Norm Transformer Block


**What & why.** One decoder layer: a residual attention sub-layer and a residual
SwiGLU sub-layer, each with RMSNorm applied to its input (pre-norm). Pre-norm is
what lets you stack dozens of layers without careful LR warmup gymnastics.

**Math.**
$$x \leftarrow x + \mathrm{Attn}(\mathrm{RMSNorm}(x))$$
$$x \leftarrow x + \mathrm{SwiGLU}(\mathrm{RMSNorm}(x))$$

**Design choices & alternatives.**
- *Post-norm* (original Transformer) normalizes after the residual add; harder to
  train deep without warmup.
- *Parallel blocks* (GPT-J, PaLM) run attention and FFN on the same normed input
  for a small speedup.

**Pitfalls.**
- Adding the residual to the *normed* input instead of the original `x`.
- Wrong weight-dict key names — note `attn.output_proj.weight`.

**Interview angles.**
- Pre-norm vs post-norm: which trains deeper and why?
- Why residual connections at all? (gradient highway, identity init)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def transformer_block(d_model, num_heads, d_ff, max_seq_len, theta, weights, in_features):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def transformer_block(d_model, num_heads, d_ff, max_seq_len, theta, weights, in_features):
    eps = 1e-5
    h = rmsnorm(d_model, eps, weights["ln1.weight"], in_features)
    attn = multihead_self_attention_with_rope(
        d_model, num_heads, max_seq_len, theta,
        weights["attn.q_proj.weight"], weights["attn.k_proj.weight"],
        weights["attn.v_proj.weight"], weights["attn.output_proj.weight"], h)
    x = in_features + attn
    h = rmsnorm(d_model, eps, weights["ln2.weight"], x)
    ffn = swiglu(d_model, d_ff, weights["ffn.w1.weight"],
                 weights["ffn.w2.weight"], weights["ffn.w3.weight"], h)
    return x + ffn


print("transformer_block defined")

### Decoder-Only Transformer LM


**What & why.** The full forward pass: embed tokens, run `num_layers` blocks, a
final RMSNorm, then an (untied) LM head projecting to vocab logits. Output is
`(batch, seq, vocab)` raw logits — never softmaxed inside the model; the loss or
the sampler decides what to do with them.

**Math.** $h_0 = \text{emb}(t)$; $h_{l+1} = \text{block}_l(h_l)$;
$\text{logits} = \mathrm{RMSNorm}(h_L)\,W_\text{head}^\top$.

**Design choices & alternatives.**
- *Tied vs untied head.* Tying `W_head` to the embedding saves parameters; we keep
  it untied so the wiring is explicit.
- *Final norm* before the head is standard in pre-norm models.

**Pitfalls.**
- Returning probabilities instead of logits.
- Off-by-one in per-layer weight keys when slicing the weight dict.

**Interview angles.**
- Why return logits, not probabilities?
- Walk through the shape of the tensor from token ids to logits.
- Where do the parameters live and how many are there roughly? (embed + layers + head)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def transformer_lm(vocab_size, context_length, d_model, num_layers, num_heads,
                   d_ff, rope_theta, weights, in_indices):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def transformer_lm(vocab_size, context_length, d_model, num_layers, num_heads,
                   d_ff, rope_theta, weights, in_indices):
    eps = 1e-5
    h = embedding(vocab_size, d_model, weights["token_embeddings.weight"], in_indices)
    for i in range(num_layers):
        lw = {
            "ln1.weight": weights[f"layers.{i}.ln1.weight"],
            "attn.q_proj.weight": weights[f"layers.{i}.attn.q_proj.weight"],
            "attn.k_proj.weight": weights[f"layers.{i}.attn.k_proj.weight"],
            "attn.v_proj.weight": weights[f"layers.{i}.attn.v_proj.weight"],
            "attn.output_proj.weight": weights[f"layers.{i}.attn.output_proj.weight"],
            "ln2.weight": weights[f"layers.{i}.ln2.weight"],
            "ffn.w1.weight": weights[f"layers.{i}.ffn.w1.weight"],
            "ffn.w2.weight": weights[f"layers.{i}.ffn.w2.weight"],
            "ffn.w3.weight": weights[f"layers.{i}.ffn.w3.weight"],
        }
        h = transformer_block(d_model, num_heads, d_ff, context_length, rope_theta, lw, h)
    h = rmsnorm(d_model, eps, weights["ln_final.weight"], h)
    return h @ weights["lm_head.weight"].transpose(-1, -2)


print("transformer_lm defined")

# 3. Tokenization

Before training we need to turn text into integer ids. We train a small byte-level
BPE tokenizer, then encode our corpus with it.

### Byte-Pair Encoding (train + encode/decode)


**What & why.** BPE starts from raw bytes (256 base tokens, so nothing is ever
out-of-vocabulary) and repeatedly merges the most frequent adjacent pair into a new
token. This gives a vocabulary that covers common words with single tokens while
still handling arbitrary text. Special tokens (e.g. `<|endoftext|>`) must never be
merged, so we split the text on them first.

**Algorithm.**
1. Pre-tokenize with a GPT-2-style regex (keeps spaces attached to words).
2. Seed the vocab with the 256 byte values plus special tokens.
3. Count adjacent byte-pair frequencies, merge the most frequent, repeat until
   `vocab_size` is reached.
4. Encoding applies the learned merges in rank order; decoding concatenates the raw
   bytes and decodes UTF-8 once at the end.

**Design choices & alternatives.**
- *WordPiece* (BERT) merges by likelihood gain rather than raw frequency.
- *Unigram / SentencePiece* starts from a big vocab and prunes.
- *Byte-level* (GPT-2) guarantees no UNK token; we use it here.

**Pitfalls.**
- Decoding each token's bytes separately — a multi-byte UTF-8 character can span a
  token boundary, so concatenate all bytes *first*, then decode with `errors="replace"`.
- Allowing a merge to cross a special-token boundary.
- Tie-breaking when two pairs have equal frequency (be deterministic).

**Interview angles.**
- Why byte-level? (no OOV, language-agnostic)
- BPE vs WordPiece vs Unigram.
- Why must decode concatenate bytes before UTF-8 decoding?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def train_bpe(input_path, vocab_size, special_tokens, **kwargs):
    raise NotImplementedError


class Tokenizer:
    def __init__(self, vocab, merges, special_tokens=None):
        raise NotImplementedError
    def encode(self, text):
        raise NotImplementedError
    def decode(self, ids):
        raise NotImplementedError

**Solution — run to continue.**

In [ ]:
GPT2_PAT = re.compile(
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


def _pretok(text):
    return GPT2_PAT.findall(text)


def _bytes_tuple(word):
    return tuple(bytes([b]) for b in word.encode("utf-8"))


def train_bpe(input_path, vocab_size, special_tokens, **kwargs):
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()
    chunks = re.split("|".join(re.escape(t) for t in special_tokens), text) if special_tokens else [text]
    counts = Counter()
    for chunk in chunks:
        for tok in _pretok(chunk):
            counts[_bytes_tuple(tok)] += 1
    vocab = {i: bytes([i]) for i in range(256)}
    nxt = 256
    for s in special_tokens or []:
        vocab[nxt] = s.encode("utf-8"); nxt += 1
    merges = []
    while len(vocab) < vocab_size:
        pairs = Counter()
        for word, c in counts.items():
            for i in range(len(word) - 1):
                pairs[(word[i], word[i + 1])] += c
        if not pairs:
            break
        best = max(pairs.items(), key=lambda kv: (kv[1], kv[0]))[0]
        merges.append(best)
        merged = best[0] + best[1]
        vocab[nxt] = merged; nxt += 1
        new_counts = Counter()
        for word, c in counts.items():
            nw, i = [], 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == best[0] and word[i + 1] == best[1]:
                    nw.append(merged); i += 2
                else:
                    nw.append(word[i]); i += 1
            new_counts[tuple(nw)] += c
        counts = new_counts
    return vocab, merges


class Tokenizer:
    def __init__(self, vocab, merges, special_tokens=None):
        self.vocab = dict(vocab)
        self.b2i = {v: k for k, v in self.vocab.items()}
        self.rank = {p: i for i, p in enumerate(merges)}
        self.special = list(special_tokens) if special_tokens else []
        for t in self.special:
            tb = t.encode("utf-8")
            if tb not in self.b2i:
                nid = max(self.vocab) + 1
                self.vocab[nid] = tb; self.b2i[tb] = nid

    def _bpe(self, word):
        syms = list(word)
        while len(syms) > 1:
            best, best_rank = None, None
            for i in range(len(syms) - 1):
                r = self.rank.get((syms[i], syms[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best = r, i
            if best is None:
                break
            syms = syms[:best] + [syms[best] + syms[best + 1]] + syms[best + 2:]
        return [self.b2i[s] for s in syms]

    def _enc_plain(self, text):
        ids = []
        for tok in _pretok(text):
            ids.extend(self._bpe(_bytes_tuple(tok)))
        return ids

    def encode(self, text):
        if not self.special:
            return self._enc_plain(text)
        pat = "|".join(re.escape(t) for t in sorted(self.special, key=len, reverse=True))
        ids, last = [], 0
        for m in re.finditer(pat, text):
            ids.extend(self._enc_plain(text[last:m.start()]))
            ids.append(self.b2i[m.group(0).encode("utf-8")])
            last = m.end()
        ids.extend(self._enc_plain(text[last:]))
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")


print("BPE defined")

# 4. Training Systems

The plumbing around the model: learning-rate schedule, gradient clipping, the
AdamW optimizer, mini-batch sampling, and checkpoint save/load.

### Cosine LR Schedule with Warmup


**What & why.** Learning rate ramps linearly from 0 to the peak over `warmup_iters`,
then decays along a cosine curve to a floor, then holds. Warmup avoids early
divergence when Adam's moment estimates are still noisy; cosine decay is the de
facto LLM schedule.

**Math.** During decay with $p = \frac{it - \text{warmup}}{\text{cycle} - \text{warmup}}$:
$$\eta = \eta_{\min} + \tfrac{1}{2}(1 + \cos(\pi p))(\eta_{\max} - \eta_{\min})$$

**Design choices & alternatives.**
- *Linear decay* / *WSD* (warmup-stable-decay) is gaining popularity for its clean
  mid-training checkpoints.
- *Constant LR* is fine for tiny runs but wastes the late phase.

**Pitfalls.**
- Division by zero when `warmup_iters == 0` or the cycle length is zero.
- LR not exactly hitting the peak at `it == warmup_iters`.

**Interview angles.**
- Why warm up the learning rate at all?
- Why cosine instead of step decay for LLMs?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def get_lr_cosine_schedule(it, max_lr, min_lr, warmup_iters, cosine_cycle_iters):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def get_lr_cosine_schedule(it, max_lr, min_lr, warmup_iters, cosine_cycle_iters):
    if it < warmup_iters:
        return max_lr * it / warmup_iters
    if it > cosine_cycle_iters:
        return min_lr
    p = (it - warmup_iters) / max(cosine_cycle_iters - warmup_iters, 1)
    return min_lr + 0.5 * (1 + math.cos(math.pi * p)) * (max_lr - min_lr)


assert abs(get_lr_cosine_schedule(10, 1.0, 0.1, 10, 100) - 1.0) < 1e-9
print("lr schedule OK")

### Global L2 Gradient Clipping


**What & why.** Compute the L2 norm of all gradients concatenated; if it exceeds a
threshold, scale every gradient down by the same factor. Prevents the occasional
huge-gradient batch from blowing up training.

**Math.** $\|g\| = \sqrt{\sum_p \sum_i g_{p,i}^2}$, then
$g_p \leftarrow g_p \cdot \min(1, \tau / (\|g\| + \epsilon))$.

**Design choices & alternatives.**
- *Per-parameter* clipping (clip each tensor independently) loses the global
  direction; global-norm clipping is standard for Transformers.
- *Value clipping* (clamp each element) is cruder and rarely used for LLMs.

**Pitfalls.**
- Including params whose `.grad` is `None`.
- Not mutating in place — the optimizer reads `p.grad` next.
- Squaring/summing across devices incorrectly in distributed setups.

**Interview angles.**
- Why global norm rather than per-tensor? (preserves update direction)
- What problem does clipping solve, and what does it *not* solve? (spikes vs bias)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def gradient_clipping(parameters, max_l2_norm):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def gradient_clipping(parameters, max_l2_norm):
    grads = [p.grad for p in parameters if p.grad is not None]
    if not grads:
        return
    total = torch.zeros((), device=grads[0].device)
    for g in grads:
        total = total + g.detach().pow(2).sum()
    total = total.sqrt()
    factor = max_l2_norm / (total + 1e-6)
    if factor < 1.0:
        for g in grads:
            g.mul_(factor)


_p = torch.nn.Parameter(torch.zeros(2)); _p.grad = torch.tensor([3.0, 4.0])
gradient_clipping([_p], 2.5)
assert abs(_p.grad.norm().item() - 2.5) < 1e-3
print("gradient clipping OK")

### AdamW Optimizer


**What & why.** Adam with *decoupled* weight decay. Adam adapts the per-parameter
step from running estimates of the gradient mean ($m$) and uncentered variance
($v$). AdamW applies weight decay directly to the parameter rather than folding it
into the gradient, which is what makes the regularization behave correctly.

**Math.** Per step $t$:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g^2$$
$$\hat\eta = \eta\,\frac{\sqrt{1-\beta_2^t}}{1-\beta_1^t}, \qquad
p \leftarrow p - \hat\eta\,\frac{m_t}{\sqrt{v_t}+\epsilon} - \eta\,\lambda\,p$$

**Design choices & alternatives.**
- *Adam* (coupled decay) ties the decay strength to the adaptive step — usually worse.
- *SGD+momentum* — cheaper memory, but Adam dominates for Transformers.
- *Muon / Adafactor / Lion* — newer optimizers; Muon orthogonalizes the update for 2-D weights.

**Pitfalls.**
- Folding weight decay into the moment update (that is plain Adam, not AdamW).
- Forgetting bias correction, especially early when $m, v$ are biased toward 0.
- Initializing state lazily on the first `step`.

**Interview angles.**
- Adam vs AdamW — what exactly is decoupled and why does it matter?
- What do $m$ and $v$ estimate, and why bias-correct them?
- Memory cost of Adam per parameter? (two extra state tensors)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        raise NotImplementedError
    def step(self, closure=None):
        raise NotImplementedError

**Solution — run to continue.**

In [ ]:
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            lr, (b1, b2) = group["lr"], group["betas"]
            eps, wd = group["eps"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g, st = p.grad, self.state[p]
                if not st:
                    st["step"] = 0
                    st["m"] = torch.zeros_like(p)
                    st["v"] = torch.zeros_like(p)
                st["step"] += 1
                t = st["step"]
                st["m"].mul_(b1).add_(g, alpha=1 - b1)
                st["v"].mul_(b2).addcmul_(g, g, value=1 - b2)
                step_lr = lr * math.sqrt(1 - b2 ** t) / (1 - b1 ** t)
                p.addcdiv_(st["m"], st["v"].sqrt().add_(eps), value=-step_lr)
                if wd != 0:
                    p.add_(p, alpha=-lr * wd)
        return loss


_x = torch.randn(8, requires_grad=True); _tgt = torch.randn(8)
_opt = AdamW([_x], lr=1e-1)
for _ in range(200):
    _opt.zero_grad(); ((_x - _tgt) ** 2).sum().backward(); _opt.step()
assert torch.allclose(_x.detach(), _tgt, atol=1e-2)
print("AdamW OK")

### Random Batch Sampling


**What & why.** Pretraining data is one long stream of token ids. A batch is just
`batch_size` random windows of length `context_length`, with the target being the
input shifted by one (next-token prediction).

**Design choices & alternatives.**
- *Memory-mapped `.bin`* files let you sample from corpora far larger than RAM.
- *Sequential / packed* sampling vs random windows — random is simplest and fine here.

**Pitfalls.**
- Sampling a start so close to the end that `start + context_length + 1` overflows.
- Returning the wrong dtype — token ids must be `long` for embedding lookup.

**Interview angles.**
- Why is `y` just `x` shifted by one? (teacher forcing / next-token objective)
- How would you sample from a dataset that does not fit in memory? (np.memmap)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def get_batch(dataset, batch_size, context_length, device):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def get_batch(dataset, batch_size, context_length, device):
    n = len(dataset)
    starts = np.random.randint(0, n - context_length, size=batch_size)
    x = np.stack([dataset[s:s + context_length] for s in starts]).astype(np.int64)
    y = np.stack([dataset[s + 1:s + 1 + context_length] for s in starts]).astype(np.int64)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)


_ds = np.arange(1000, dtype=np.int32)
_x, _y = get_batch(_ds, 4, 8, "cpu")
assert (_y == _x + 1).all()
print("get_batch OK", _x.shape)

### Checkpoint Save / Load


**What & why.** To resume training you must persist enough state to continue
exactly: model weights, optimizer state (the Adam moments and step counters), and
the iteration number. Saving only the model loses the optimizer's momentum.

**Design choices & alternatives.**
- *Sharded / distributed* checkpoints for large models.
- *`safetensors`* for safe, fast weight-only serialization in production.

**Pitfalls.**
- Forgetting optimizer state — resuming then behaves like a fresh optimizer.
- Device mismatch on load; `map_location="cpu"` is the portable default.

**Interview angles.**
- What must a checkpoint contain to resume *bit-for-bit*?
- Why save optimizer state and not just weights?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def save_checkpoint(model, optimizer, iteration, out):
    raise NotImplementedError
def load_checkpoint(src, model, optimizer):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def save_checkpoint(model, optimizer, iteration, out):
    torch.save({"model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iteration": iteration}, out)


def load_checkpoint(src, model, optimizer):
    state = torch.load(src, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    return int(state["iteration"])


_m = torch.nn.Linear(8, 4); _o = torch.optim.AdamW(_m.parameters())
_buf = io.BytesIO(); save_checkpoint(_m, _o, 42, _buf); _buf.seek(0)
_m2 = torch.nn.Linear(8, 4); _o2 = torch.optim.AdamW(_m2.parameters())
assert load_checkpoint(_buf, _m2, _o2) == 42
print("checkpointing OK")

# 5. Train a Tiny Model End to End

Everything above now comes together. We:

1. define a tiny built-in corpus,
2. train a small BPE tokenizer on it and encode it to a token stream,
3. build a `TinyLM` module whose parameters match the `transformer_lm` weight-dict
   layout,
4. run a real training loop with our own AdamW, cosine schedule, and gradient
   clipping,
5. checkpoint the result.

It is intentionally small so it runs on CPU in well under a minute. The loss going
down is the proof that every block above is wired correctly.

In [ ]:
CORPUS = (
    "the quick brown fox jumps over the lazy dog. "
    "a journey of a thousand miles begins with a single step. "
    "to be or not to be, that is the question. "
    "all that glitters is not gold; all who wander are not lost. "
    "the only thing we have to fear is fear itself. "
    "i think, therefore i am. knowledge is power and power corrupts. "
) * 40

# Write the corpus to a temp file so train_bpe can read it like a real corpus.
import tempfile, os
_tmp = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8")
_tmp.write(CORPUS); _tmp.close()

VOCAB_SIZE = 320  # 256 bytes + a handful of merges
vocab, merges = train_bpe(_tmp.name, VOCAB_SIZE, special_tokens=["<|endoftext|>"])
os.unlink(_tmp.name)
tokenizer = Tokenizer(vocab, merges, special_tokens=["<|endoftext|>"])

ids = np.array(tokenizer.encode(CORPUS), dtype=np.int32)
print("vocab size:", len(vocab), "| merges:", len(merges), "| corpus tokens:", len(ids))
print("round-trip OK:", tokenizer.decode(tokenizer.encode("the lazy dog")) == "the lazy dog")

### TinyLM module

`transformer_lm` takes a plain dict of weight tensors. To *train*, those tensors
need to be `nn.Parameter`s that an optimizer can see. `TinyLM` holds a
`ParameterDict` keyed exactly the way `transformer_lm` expects, initializes them,
and calls our pure-function model in `forward`.

In [ ]:
class TinyLM(torch.nn.Module):
    def __init__(self, vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta):
        super().__init__()
        self.cfg = dict(vocab_size=vocab_size, context_length=context_length, d_model=d_model,
                        num_layers=num_layers, num_heads=num_heads, d_ff=d_ff, rope_theta=rope_theta)
        p = {}
        def param(*shape, scale):
            return torch.nn.Parameter(torch.randn(*shape) * scale)
        s = d_model ** -0.5
        p["token_embeddings.weight"] = param(vocab_size, d_model, scale=s)
        for i in range(num_layers):
            p[f"layers.{i}.ln1.weight"] = torch.nn.Parameter(torch.ones(d_model))
            p[f"layers.{i}.ln2.weight"] = torch.nn.Parameter(torch.ones(d_model))
            for proj in ("q_proj", "k_proj", "v_proj", "output_proj"):
                p[f"layers.{i}.attn.{proj}.weight"] = param(d_model, d_model, scale=s)
            p[f"layers.{i}.ffn.w1.weight"] = param(d_ff, d_model, scale=s)
            p[f"layers.{i}.ffn.w2.weight"] = param(d_model, d_ff, scale=d_ff ** -0.5)
            p[f"layers.{i}.ffn.w3.weight"] = param(d_ff, d_model, scale=s)
        p["ln_final.weight"] = torch.nn.Parameter(torch.ones(d_model))
        p["lm_head.weight"] = param(vocab_size, d_model, scale=s)
        # ParameterDict keys cannot contain '.', so escape them and unescape on use.
        self.params = torch.nn.ParameterDict({k.replace(".", "__"): v for k, v in p.items()})

    def weights(self):
        return {k.replace("__", "."): v for k, v in self.params.items()}

    def forward(self, in_indices):
        return transformer_lm(weights=self.weights(), in_indices=in_indices, **self.cfg)


CONTEXT_LEN = 32
model = TinyLM(vocab_size=len(vocab), context_length=CONTEXT_LEN, d_model=64,
               num_layers=2, num_heads=4, d_ff=128, rope_theta=10000.0)
n_params = sum(p.numel() for p in model.parameters())
print(f"TinyLM params: {n_params:,}")

### The training loop

Standard next-token training: sample a batch, forward, cross-entropy on the
flattened logits/targets, backward, clip gradients, set the scheduled LR, step
AdamW. We log the loss so you can watch it fall.

In [ ]:
BATCH_SIZE, STEPS = 16, 300
MAX_LR, MIN_LR, WARMUP = 3e-3, 3e-4, 30

opt = AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.01)
losses = []
for step in range(STEPS):
    x, y = get_batch(ids, BATCH_SIZE, CONTEXT_LEN, "cpu")
    logits = model(x)                       # (B, T, V)
    loss = cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    gradient_clipping(model.parameters(), max_l2_norm=1.0)
    lr = get_lr_cosine_schedule(step, MAX_LR, MIN_LR, WARMUP, STEPS)
    for g in opt.param_groups:
        g["lr"] = lr
    opt.step()
    losses.append(loss.item())
    if step % 50 == 0 or step == STEPS - 1:
        print(f"step {step:3d}  lr {lr:.4f}  loss {loss.item():.4f}")

print(f"\nloss: {losses[0]:.3f} -> {losses[-1]:.3f} (should drop a lot)")
assert losses[-1] < losses[0], "loss did not decrease - check your blocks"

# Checkpoint it.
ckpt = io.BytesIO()
save_checkpoint(model, opt, STEPS, ckpt)
print("checkpoint bytes:", ckpt.getbuffer().nbytes)

Optional: plot the loss curve (skipped silently if matplotlib is unavailable).

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 3))
    plt.plot(losses)
    plt.xlabel("step"); plt.ylabel("cross-entropy loss"); plt.title("TinyLM training")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("plot skipped:", e)

# 6. Inference

A trained model is only useful if you can sample from it. We build the KV cache
(the key inference optimization), the sampling filters, the generation loop, and
then generate text from the model we just trained.

### KV Cache


**What & why.** During generation we feed one token at a time. Recomputing K and V
for the entire prefix at every step is $O(n^2)$ wasted work, so we cache the K/V of
all previous positions in a pre-allocated buffer and only compute K/V for the new
token. This is the single most important inference optimization.

**Layout.** A buffer of shape `(num_layers, max_batch, num_heads, max_seq_len, head_dim)`
per K and V, plus a per-layer write position.

**Design choices & alternatives.**
- *Paged attention* (vLLM) manages the cache in fixed blocks to avoid fragmentation.
- *MQA / GQA* shrink the cache by sharing K/V across heads — the main reason they exist.
- *Quantized KV cache* trades a little quality for a much smaller footprint.

**Pitfalls.**
- Forgetting to advance the write position after each step.
- Returning the whole buffer instead of the valid `[0:length)` slice.

**Interview angles.**
- Why does the KV cache help — what is recomputed without it?
- How large is the KV cache, and how do MQA/GQA reduce it?
- Why does the cache make decode memory-bandwidth bound?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
class KVCache:
    def __init__(self, num_layers, max_batch_size, max_seq_len, num_heads, head_dim,
                 dtype=torch.float32, device="cpu"):
        raise NotImplementedError
    def reset(self, batch_size):
        raise NotImplementedError
    def update(self, layer_idx, k_step, v_step):
        raise NotImplementedError

**Solution — run to continue.**

In [ ]:
class KVCache:
    def __init__(self, num_layers, max_batch_size, max_seq_len, num_heads, head_dim,
                 dtype=torch.float32, device="cpu"):
        self.num_layers = num_layers
        shape = (num_layers, max_batch_size, num_heads, max_seq_len, head_dim)
        self.k_buf = torch.zeros(shape, dtype=dtype, device=device)
        self.v_buf = torch.zeros(shape, dtype=dtype, device=device)
        self.lengths = [0] * num_layers

    def reset(self, batch_size):
        self.lengths = [0] * self.num_layers

    def update(self, layer_idx, k_step, v_step):
        b, h, step_len, d = k_step.shape
        start = self.lengths[layer_idx]
        end = start + step_len
        self.k_buf[layer_idx, :b, :, start:end, :] = k_step
        self.v_buf[layer_idx, :b, :, start:end, :] = v_step
        self.lengths[layer_idx] = end
        return self.k_buf[layer_idx, :b, :, :end, :], self.v_buf[layer_idx, :b, :, :end, :], end


_c = KVCache(2, 1, 8, 2, 4); _c.reset(1)
_, _, _l1 = _c.update(0, torch.randn(1, 2, 3, 4), torch.randn(1, 2, 3, 4))
_, _, _l2 = _c.update(0, torch.randn(1, 2, 1, 4), torch.randn(1, 2, 1, 4))
assert (_l1, _l2) == (3, 4)
print("KVCache OK")

### Sampling: Temperature, Top-K, Top-P


**What & why.** Given next-token logits, decide which token to emit. Temperature
sharpens ($T<1$) or flattens ($T>1$) the distribution; top-k keeps only the k most
likely tokens; top-p (nucleus) keeps the smallest set whose cumulative probability
exceeds p. Filtering trims the long tail of low-probability junk.

**Math.** Temperature softmax $p_i = e^{z_i/T} / \sum_j e^{z_j/T}$, then a
multinomial draw. Filters set rejected logits to $-\infty$ before the softmax.

**Design choices & alternatives.**
- *Greedy* ($T=0$, argmax) — deterministic, repetitive.
- *Top-p vs top-k.* Top-p adapts the candidate set to the distribution's shape;
  top-k is a fixed cutoff. They compose.
- *min-p*, *typical*, *repetition penalty* — further refinements.

**Pitfalls.**
- Filtering after the softmax instead of before.
- Top-p removing everything — always keep at least the top token.
- Composition order: filter (top-k then top-p), then temperature softmax, then sample.

**Interview angles.**
- Top-k vs top-p — when does each fail? (top-k too many/few; top-p tail sensitivity)
- What does temperature do to entropy?
- Why mask to $-\infty$ rather than zeroing probabilities?

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def sample(logits, temperature=1.0):
    raise NotImplementedError
def top_k_filter(logits, k):
    raise NotImplementedError
def top_p_filter(logits, p):
    raise NotImplementedError
def sample_token(logits, temperature=1.0, top_k=None, top_p=None):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def sample(logits, temperature=1.0):
    if temperature <= 0:
        return logits.argmax(dim=-1)
    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)


def top_k_filter(logits, k):
    if k is None or k <= 0 or k >= logits.shape[-1]:
        return logits
    vals, _ = torch.topk(logits, k=k, dim=-1)
    thresh = vals[..., -1:].expand_as(logits)
    return torch.where(logits < thresh, torch.full_like(logits, float("-inf")), logits)


def top_p_filter(logits, p):
    if p is None or p >= 1.0 or p <= 0.0:
        return logits
    s_logits, s_idx = torch.sort(logits, dim=-1, descending=True)
    cum = torch.softmax(s_logits, dim=-1).cumsum(dim=-1)
    remove_sorted = (cum - torch.softmax(s_logits, dim=-1)) >= p
    remove_sorted[..., 0] = False
    remove = torch.zeros_like(remove_sorted)
    remove.scatter_(-1, s_idx, remove_sorted)
    return torch.where(remove, torch.full_like(logits, float("-inf")), logits)


def sample_token(logits, temperature=1.0, top_k=None, top_p=None):
    if top_k is not None:
        logits = top_k_filter(logits, top_k)
    if top_p is not None:
        logits = top_p_filter(logits, top_p)
    return sample(logits, temperature=temperature)


torch.manual_seed(0)
_l = torch.tensor([[0.0, 0.0, 10.0]])
_s = torch.cat([sample_token(_l, temperature=1.0) for _ in range(50)])
assert (_s == 2).float().mean() > 0.9
print("sampling OK")

### Autoregressive Generation Loop


**What & why.** Tie it together: run the prompt through the model (prefill), then
repeatedly sample the next token and feed it back until `max_new_tokens` or an EOS
token. Returns only the continuation.

**Design choices & alternatives.**
- *Beam search* keeps several hypotheses; standard for translation, rare for chat LLMs.
- *Speculative decoding* uses a small draft model to propose tokens a big model verifies.

**Pitfalls.**
- Returning the prompt along with the continuation.
- Mismanaging `position_offset` so RoPE positions drift.

**Interview angles.**
- Prefill vs decode — why are their compute profiles different? (parallel vs sequential)
- How would you batch generations of different lengths? (padding + attention mask)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def generate(step_fn, prompt_ids, max_new_tokens, eos_token_id=None,
             temperature=1.0, top_k=None, top_p=None):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def generate(step_fn, prompt_ids, max_new_tokens, eos_token_id=None,
             temperature=1.0, top_k=None, top_p=None):
    if prompt_ids.dim() == 1:
        prompt_ids = prompt_ids.unsqueeze(0)
    logits = step_fn(prompt_ids, 0)
    last = logits[:, -1, :]
    generated = []
    for step in range(max_new_tokens):
        nxt = sample_token(last, temperature=temperature, top_k=top_k, top_p=top_p)
        generated.append(nxt)
        if eos_token_id is not None and (nxt == eos_token_id).all():
            break
        logits = step_fn(nxt.unsqueeze(-1), prompt_ids.shape[1] + step)
        last = logits[:, -1, :]
    return torch.stack(generated, dim=-1)


def _toy_step(token_ids, position_offset):
    logits = torch.full((*token_ids.shape, 10), -1e9)
    logits[..., 4] = 0.0
    return logits


assert generate(_toy_step, torch.tensor([[0, 0]]), 5, temperature=0.0).tolist() == [[4, 4, 4, 4, 4]]
print("generate OK")

### Generate from the model we trained

Our `transformer_lm` recomputes the full sequence each call (no KV cache wired into
it), so the `step_fn` below keeps the running token history and feeds the whole
sequence to the model, returning logits for the new positions. That satisfies the
`generate` interface and is correct; a production model would instead thread the
`KVCache` through attention. We greedily continue a prompt and decode it.

In [ ]:
@torch.no_grad()
def make_step_fn(model, context_len):
    history = {"ids": None}
    def step_fn(token_ids, position_offset):
        if position_offset == 0:
            history["ids"] = token_ids
        else:
            history["ids"] = torch.cat([history["ids"], token_ids], dim=1)
        seq = history["ids"][:, -context_len:]
        logits = model(seq)                      # (B, T, V)
        # Return logits aligned so that [:, -1, :] is the next-token distribution.
        return logits[:, -token_ids.shape[1]:, :] if position_offset else logits
    return step_fn


prompt = "the quick brown"
prompt_ids = torch.tensor([tokenizer.encode(prompt)])
cont = generate(make_step_fn(model, CONTEXT_LEN), prompt_ids,
                max_new_tokens=40, temperature=0.8, top_k=20)
print("prompt:     ", repr(prompt))
print("continuation:", repr(tokenizer.decode(cont[0].tolist())))

# 7. Alignment (Bonus)

Pretraining gives a next-token predictor. Alignment makes it follow instructions
and prefer good answers. Two core losses below: supervised fine-tuning with prompt
masking, and DPO for learning from preference pairs.

### SFT Loss with Prompt Masking


**What & why.** In SFT we pack `(prompt + response)` into one sequence and train
next-token prediction, but only the **response** tokens should contribute to the
loss. Counting prompt tokens both wastes gradient and teaches the model to generate
the user's prompt. A 0/1 mask zeroes the prompt positions.

**Math.**
$$\mathcal{L} = \frac{\sum_{b,t} m_{b,t}\,\mathrm{CE}(z_{b,t}, y_{b,t})}{\max(\sum_{b,t} m_{b,t},\,1)}$$

**Design choices & alternatives.**
- *Mask the prompt* (here) vs *train on everything* — masking is standard for chat SFT.
- *Loss on completion only* vs *weighted* schemes for multi-turn data.

**Pitfalls.**
- Dividing by the total token count instead of the number of unmasked tokens.
- An all-zero mask → division by zero; clamp the denominator at 1.

**Interview angles.**
- Why mask the prompt in SFT?
- How does this differ from plain pretraining cross-entropy? (the mask + normalization)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def sft_loss(logits, targets, loss_mask):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def sft_loss(logits, targets, loss_mask):
    b, t, v = logits.shape
    per_token = F.cross_entropy(logits.reshape(b * t, v), targets.reshape(b * t), reduction="none")
    mask = loss_mask.reshape(b * t).to(per_token.dtype)
    return (per_token * mask).sum() / mask.sum().clamp_min(1.0)


_lg = torch.randn(2, 5, 10); _tg = torch.randint(0, 10, (2, 5))
_mk = torch.zeros(2, 5); _mk[:, -2:] = 1
_ref = F.cross_entropy(_lg[:, -2:, :].reshape(-1, 10), _tg[:, -2:].reshape(-1))
assert torch.allclose(sft_loss(_lg, _tg, _mk), _ref, atol=1e-5)
print("sft_loss OK")

### DPO Preference Loss


**What & why.** Direct Preference Optimization learns from `(prompt, chosen,
rejected)` pairs without training a separate reward model or running PPO. It needs
only the summed log-probabilities of each response under the current policy and a
frozen reference model, and pushes the policy to prefer chosen over rejected while
the reference anchors it from drifting.

**Math.** With $h = \log\pi_\theta(y) - \log\pi_\text{ref}(y)$:
$$\mathcal{L} = -\mathbb{E}\big[\log\sigma(\beta(h_w - h_l))\big]$$
Implicit rewards $r = \beta h$ are returned for monitoring.

**Design choices & alternatives.**
- *PPO / RLHF* — train a reward model, then RL; more moving parts, more compute.
- *IPO, KTO, ORPO* — DPO variants with different objectives or data assumptions.
- *$\beta$* controls how far the policy may move from the reference.

**Pitfalls.**
- Using `log(sigmoid(x))` instead of the stable `logsigmoid`.
- Letting gradients flow into the reference model (it is frozen).
- Returning rewards that still carry gradient — detach them.

**Interview angles.**
- What does DPO replace in the RLHF pipeline, and why is it simpler?
- Role of the reference model and of $\beta$?
- Derive the loss from the Bradley-Terry preference model. (optional, deep)

**Your turn.** Fill in the cell below, then run it. If you get stuck, run the Solution cell that follows to keep the end-to-end pipeline working.

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    raise NotImplementedError

**Solution — run to continue.**

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    h_chosen = policy_chosen_logps - ref_chosen_logps
    h_rejected = policy_rejected_logps - ref_rejected_logps
    loss = -F.logsigmoid(beta * (h_chosen - h_rejected)).mean()
    return loss, (beta * h_chosen).detach(), (beta * h_rejected).detach()


_b = 4
_loss, _cr, _rr = dpo_loss(torch.zeros(_b), torch.zeros(_b), torch.zeros(_b), torch.zeros(_b), beta=0.1)
assert abs(_loss.item() - math.log(2)) < 1e-5   # policy == ref -> -log sigma(0) = log 2
print("dpo_loss OK  (loss = log 2 =", round(math.log(2), 4), ")")

# Done

You built every component of a decoder-only Transformer LM, trained a tiny one on
real text, sampled from it, and implemented the two core alignment losses. The
same blocks, scaled up with more data, layers, and width, are exactly what runs in
production LLMs.

**Where to go next**
- Wire the `KVCache` directly through attention so decode is truly incremental.
- Swap AdamW for the Muon optimizer on the 2-D weight matrices.
- Train on a larger corpus with a memory-mapped `.bin` and a bigger vocab.
- Add an SFT pass on instruction pairs, then a DPO pass on preference pairs.